<a href="https://colab.research.google.com/github/JesseOrtega01/streamlit1/blob/main/RetoEmpleados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reto: Desercion de empleados por Jesse Roberto Peña Ortega:

1.-Importar las librerias requeridas

In [141]:
import pandas as pd
from datetime import datetime,date

2.-Lee el archivo CSV y guardar en un DF llamado EmpleadosAttrition

In [142]:
EmpleadosAttrition=pd.read_csv('empleadosRETO.csv')

3.-Elimina las columnas (estimacion propia) que no tienen relacion con la salida

In [143]:
EmpleadosAttrition=EmpleadosAttrition.drop(['EmployeeCount','Over18','StandardHours','EmployeeNumber'],axis=1)

4.-Calculo de variables adicionales
5.-Columna llamada "Year" para  año apartor de que fueron contratados
6.-crear columna "YearsAtCompany" para saber cuantos años llevan en la compañia hasta 2018


In [144]:
EmpleadosAttrition.columns

Index(['Age', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education',
       'EducationField', 'EnvironmentSatisfaction', 'Gender', 'JobInvolvement',
       'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus',
       'MonthlyIncome', 'NumCompaniesWorked', 'HiringDate', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsInCurrentRole', 'YearsSinceLastPromotion', 'Attrition'],
      dtype='object')

se eliminan los valores erroneos para la fecha (ejemplo:30 de febrero)

In [145]:
EmpleadosAttrition['HiringDate']=pd.to_datetime(EmpleadosAttrition['HiringDate'],errors='coerce')
EmpleadosAttrition.dropna(subset=['HiringDate'],inplace=True)

In [146]:
#EmpleadosAttrition['HiredYear']=EmpleadosAttrition['HiringDate'].str.split('/').str[0].astype(int)
EmpleadosAttrition['Year']=pd.DatetimeIndex(EmpleadosAttrition['HiringDate']).year
print(EmpleadosAttrition['Year'])

0      2013
1      2015
2      2017
3      2010
4      2011
       ... 
395    2013
396    2016
397    2008
398    2018
399    2010
Name: Year, Length: 399, dtype: int32


si se tuviera que hacer el calculo al dia de hoy el calculo seria el siguiente:

In [147]:
ThisYear=datetime.now().year
EmpleadosAttrition['TimeAtCompany']=ThisYear-EmpleadosAttrition['Year']
print(EmpleadosAttrition['TimeAtCompany'])

0      13
1      11
2       9
3      16
4      15
       ..
395    13
396    10
397    18
398     8
399    16
Name: TimeAtCompany, Length: 399, dtype: int32


Haciendolo considerando solo a 2018

In [148]:
EmpleadosAttrition['YearsAtCompany']=2018-EmpleadosAttrition['Year']
print(EmpleadosAttrition['YearsAtCompany'])

0       5
1       3
2       1
3       8
4       7
       ..
395     5
396     2
397    10
398     0
399     8
Name: YearsAtCompany, Length: 399, dtype: int32


7/8/9.-DistanceFromHome en entero y renombrado a DistanceFromHome_KM

In [149]:
EmpleadosAttrition.rename(columns={'DistanceFromHome':'DistanceFromHome_KM'}, inplace=True)
print('Original \n',EmpleadosAttrition['DistanceFromHome_KM'])
EmpleadosAttrition['DistanceFromHome']=EmpleadosAttrition['DistanceFromHome_KM'].str.replace(' km','')
EmpleadosAttrition['DistanceFromHome']=EmpleadosAttrition['DistanceFromHome'].astype(int)
print('Nuevo \n',EmpleadosAttrition['DistanceFromHome'])

Original 
 0       1 km
1       6 km
2       7 km
3       7 km
4      15 km
       ...  
395    14 km
396    20 km
397    11 km
398     4 km
399    14 km
Name: DistanceFromHome_KM, Length: 399, dtype: object
Nuevo 
 0       1
1       6
2       7
3       7
4      15
       ..
395    14
396    20
397    11
398     4
399    14
Name: DistanceFromHome, Length: 399, dtype: int64


10.-Borrar columnas ya no usadas:Year, HiringDate y DistanceFromHome_km

In [150]:
EmpleadosAttrition=EmpleadosAttrition.drop(['Year','HiringDate','DistanceFromHome_KM'],axis=1)
print(EmpleadosAttrition.columns)

Index(['Age', 'BusinessTravel', 'Department', 'Education', 'EducationField',
       'EnvironmentSatisfaction', 'Gender', 'JobInvolvement', 'JobLevel',
       'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome',
       'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'TotalWorkingYears',
       'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'Attrition', 'TimeAtCompany',
       'YearsAtCompany', 'DistanceFromHome'],
      dtype='object')


11.-nuevo DF  llamado "SueldoPromedio" que tenga el MonthlyIncome promedio por departamento de empleado y guardarlo en la variable llamada SueldoPromedio

In [151]:
SueldoPromedio=(EmpleadosAttrition.groupby('Department')['MonthlyIncome'].mean().reset_index())
print(SueldoPromedio)

               Department  MonthlyIncome
0         Human Resources    6239.888889
1  Research & Development    6804.149813
2                   Sales    7192.609756


12.-Escalar MonhtlyIncome a valores entre 0 y 1 (escalamiento Min Max)

In [152]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
EmpleadosAttrition['MonthlyIncome'] = scaler.fit_transform(EmpleadosAttrition[['MonthlyIncome']])
print(EmpleadosAttrition['MonthlyIncome'])

0      0.864269
1      0.207340
2      0.088062
3      0.497574
4      0.664470
         ...   
395    0.075248
396    0.187197
397    0.589327
398    0.121124
399    0.092122
Name: MonthlyIncome, Length: 399, dtype: float64


13.-convertir variables categoricas a numericas

In [153]:
from pandas.core.arrays import sparse
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False,drop='first')
encoded_columns = pd.DataFrame(encoder.fit_transform(EmpleadosAttrition[['BusinessTravel','Department',
                                                                         'EducationField','Gender','JobRole',
                                                                         'MaritalStatus','OverTime', 'Attrition']]))
encoded_columns.columns = encoder.get_feature_names_out(['BusinessTravel','Department',
                                                                         'EducationField','Gender','JobRole',
                                                                         'MaritalStatus','OverTime', 'Attrition'])
EmpleadosAttrition = pd.concat([EmpleadosAttrition, encoded_columns], axis=1)
EmpleadosAttrition = EmpleadosAttrition.drop(['BusinessTravel','Department',
                                                                         'EducationField','Gender','JobRole',
                                                                         'MaritalStatus','OverTime',
                                              'Attrition'], axis=1)
EmpleadosAttrition.head()

,Age,Education,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,...,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,MaritalStatus_nan,OverTime_Yes,Attrition_Yes
0,50.0,2.0,4.0,3.0,4.0,4.0,0.864269,9.0,22.0,4.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,36.0,2.0,2.0,3.0,2.0,2.0,0.207340,6.0,20.0,4.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,21.0,1.0,2.0,3.0,1.0,2.0,0.088062,1.0,13.0,3.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,52.0,4.0,2.0,3.0,3.0,2.0,0.497574,7.0,19.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,33.0,1.0,2.0,3.0,3.0,3.0,0.664470,7.0,12.0,3.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0


14.-evaluacion de variables. calculo de correlacion lineal con respecto a Attrition

In [154]:
EmpleadosAttrition.corrwith(EmpleadosAttrition['Attrition_Yes'])


,0
Age,-0.180356
Education,-0.023176
EnvironmentSatisfaction,-0.029108
JobInvolvement,-0.051565
JobLevel,-0.186084
JobSatisfaction,-0.100914
MonthlyIncome,-0.142190
NumCompaniesWorked,-0.049956
PercentSalaryHike,-0.016926
PerformanceRating,-0.039085


15.- seleccionar solo aquellas que tenga una relacion mayoe o igual a 0.1 guardandola en un DF llamado "EmpleadosAttritionFinal"

In [155]:
print(EmpleadosAttrition.corrwith(EmpleadosAttrition['Attrition_Yes']).sort_values(ascending=False)>=0.1 )

Attrition_Yes                         True
OverTime_Yes                          True
MaritalStatus_Single                  True
JobRole_Sales Representative          True
EducationField_Technical Degree       True
JobRole_Laboratory Technician         True
WorkLifeBalance                      False
Department_Sales                     False
BusinessTravel_Travel_Rarely         False
BusinessTravel_Travel_Frequently     False
JobRole_Human Resources              False
TrainingTimesLastYear                False
EducationField_Marketing             False
MaritalStatus_nan                    False
JobRole_Research Scientist           False
JobRole_Sales Executive              False
EducationField_Other                 False
RelationshipSatisfaction             False
PercentSalaryHike                    False
Education                            False
EducationField_Life Sciences         False
EnvironmentSatisfaction              False
Gender_Male                          False
DistanceFro

In [156]:
ratio=EmpleadosAttrition.corrwith(EmpleadosAttrition['Attrition_Yes'])
col_select=ratio[ratio>=0.1]
EmpleadosAttritionFinal=EmpleadosAttrition[col_select.index]
EmpleadosAttritionFinal.head()


,EducationField_Technical Degree,JobRole_Laboratory Technician,JobRole_Sales Representative,MaritalStatus_Single,OverTime_Yes,Attrition_Yes
0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,1.0,0.0,1.0
3,0.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,1.0


16.-Nueva variable llamada "EmpleadosAttritionPCA"  formada por los principales componentes del frame "EmpleadosAttritionFinal",

In [157]:
from sklearn.decomposition import PCA
pca = PCA()
EmpleadosAttritionFinal=EmpleadosAttritionFinal.dropna()
EmpleadosAttritionPCA = pca.fit_transform(EmpleadosAttritionFinal)
EmpleadosAttritionPCA

array([[-0.41938778, -0.02559025, -0.14026445,  0.08287468, -0.09308544,
        -0.00919999],
       [-0.41938778, -0.02559025, -0.14026445,  0.08287468, -0.09308544,
        -0.00919999],
       [ 0.73112396,  0.77060748,  0.13342795,  0.84655022,  0.26101282,
        -0.51802065],
       ...,
       [-0.42230906, -0.15141262,  0.7817731 , -0.22723666,  0.09715494,
        -0.049777  ],
       [-0.41938778, -0.02559025, -0.14026445,  0.08287468, -0.09308544,
        -0.00919999],
       [ 0.64508137,  0.65589034,  0.17246615,  0.51276953, -0.41240985,
         0.12463402]])

17.-Agrega el mínimo número de Componentes Principales en columnas del frame EmpleadosAttritionPCA que logren explicar el 80% de la varianza, al frame EmpleadosAttritionFinal

In [158]:
pca_80=PCA(n_components=0.8)
EmpleadosAttritionPCA=pca_80.fit_transform(EmpleadosAttritionFinal)
columnas= [f"PC{i+1}" for i in range(EmpleadosAttritionPCA.shape[1])]
EmpleadosAttritionPCA=pd.DataFrame(EmpleadosAttritionPCA,columns=columnas)
EmpleadosAttritionPCA

,PC1,PC2,PC3,PC4
0,-0.419388,-0.025590,-0.140264,0.082875
1,-0.419388,-0.025590,-0.140264,0.082875
2,0.731124,0.770607,0.133428,0.846550
3,0.128543,0.755029,-0.129252,-0.202312
4,0.748556,-0.715802,-0.076536,0.413148
...,...,...,...,...
394,0.748556,-0.715802,-0.076536,0.413148
395,0.232018,-0.616662,-0.378254,-0.301934
396,-0.422309,-0.151413,0.781773,-0.227237
397,-0.419388,-0.025590,-0.140264,0.082875


In [159]:
EmpleadosAttritionFinal=pd.concat([EmpleadosAttritionFinal,EmpleadosAttritionPCA],axis=1)
print(EmpleadosAttritionFinal)

     EducationField_Technical Degree  JobRole_Laboratory Technician  \
0                                0.0                            0.0   
1                                0.0                            0.0   
2                                0.0                            0.0   
3                                0.0                            0.0   
4                                0.0                            0.0   
..                               ...                            ...   
395                              0.0                            0.0   
396                              0.0                            0.0   
397                              0.0                            1.0   
398                              0.0                            0.0   
229                              0.0                            0.0   

     JobRole_Sales Representative  MaritalStatus_Single  OverTime_Yes  \
0                             0.0                   0.0           0.0   
1

In [160]:
import plotly.express as px
import pandas as pd

fig=px.scatter_3d(EmpleadosAttritionFinal,x='PC1',y='PC2',z='PC4',color='Attrition_Yes')
fig.show()

18.-Guarda el set de datos que has formado y que tienes en EmpleadosAttritionFinal en un archivo CSV llamado EmpleadosAttritionFinal.csv.

In [161]:
EmpleadosAttritionFinal.to_csv('EmpleadosAttritionFinal.csv',index=False)
